# GPU Memory Hierarchy & Data Movement with CuTe DSL

A modern NVIDIA GPU has **5 levels of memory**, each trading off size for speed:

```
                    Scope         Capacity (typical)    Latency        Bandwidth
  +-----------+
  |   RMEM    |     Per-thread    255 registers/thread  ~0 cycles      ~unlimited
  +-----------+
  | L1 Cache  |     Per-SM        128-256 KB            ~30 cycles     hardware-managed
  +-----------+
  |   SMEM    |     Per-block     up to 228 KB (Ada)    ~20-30 cycles  ~19 TB/s (aggregate)
  +-----------+
  | L2 Cache  |     Device-wide   up to 96 MB (Ada)     ~200 cycles    hardware-managed
  +-----------+
  |   GMEM    |     Device-wide   up to 80 GB (A100)    ~400 cycles    ~2 TB/s
  +-----------+
```

**Three of these are programmer-controlled:** Global Memory (GMEM), Shared Memory (SMEM), and Register Memory (RMEM). The L1 and L2 caches are hardware-managed -- data flows through them automatically, though we can influence their behavior with cache hints.

The fundamental job of a high-performance GPU kernel is to **move data up this hierarchy** as efficiently as possible: from GMEM into SMEM (shared across a thread block), then from SMEM into RMEM (private to each thread), where the actual computation happens.

CuTe DSL provides **CopyAtoms** -- abstractions over the hardware copy instructions -- to express these data movement patterns. This notebook explores each memory level and demonstrates how to move data between them.

**Authors: Claude Code & Pramodith**

In [ ]:
%pip install -q torch triton nvidia-cutlass nvidia-cutlass-dsl

In [ ]:
import os
import torch
import triton

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

assert torch.cuda.is_available(), "CUDA GPU required"
major, minor = torch.cuda.get_device_capability()
os.environ["CUTE_DSL_ARCH"] = f"sm_{major}{minor}" + ("a" if major >= 9 else "")

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute capability: sm_{major}{minor}")
print(f"Target arch: {os.environ['CUTE_DSL_ARCH']}")

## The 5 Memory Levels in Detail

### 1. Global Memory (GMEM)
- **Scope:** Visible to all threads on the device
- **Backed by:** HBM (High Bandwidth Memory) or GDDR
- **Size:** GBs (e.g., 12 GB on RTX 4070, 80 GB on A100)
- **Latency:** ~400-800 cycles
- **Access pattern:** Coalesced 128-byte transactions. When 32 threads in a warp access consecutive addresses, the hardware merges them into minimal transactions. Scattered access wastes bandwidth.

This is where your PyTorch tensors live. Every kernel starts and ends by reading from and writing to GMEM.

### 2. L2 Cache
- **Scope:** Device-wide (shared by all SMs)
- **Size:** MBs (e.g., 36 MB on RTX 4070, 40 MB on A100)
- **Latency:** ~200 cycles
- **Managed by:** Hardware (transparent caching of GMEM accesses)
- **Programmer influence:** Cache eviction hints (`EVICT_FIRST`, `EVICT_LAST`, `EVICT_NORMAL`) via CopyAtom parameters, and L2 persistence controls via `cudaAccessPolicyWindow`

### 3. Shared Memory (SMEM)
- **Scope:** Per-thread-block (all threads in a block see the same SMEM)
- **Size:** Configurable per-block, up to 228 KB on Ada/Hopper
- **Latency:** ~20-30 cycles
- **Key feature:** Software-managed scratchpad. The programmer explicitly allocates and fills it.
- **Why it matters:** When multiple threads in a block need the same data, loading it once into SMEM and reading it N times is far cheaper than N separate GMEM loads.

### 4. L1 Cache
- **Scope:** Per-SM
- **Size:** 128-256 KB (shared/configurable with SMEM on some architectures)
- **Latency:** ~30 cycles
- **Managed by:** Hardware (caches GMEM and register spills)
- **Note:** On modern GPUs (Ampere+), L1 and SMEM share the same on-chip SRAM. The split is configurable via `cudaFuncSetAttribute`.

### 5. Register Memory (RMEM)
- **Scope:** Per-thread (completely private)
- **Size:** Up to 255 32-bit registers per thread
- **Latency:** ~0 cycles (operands are directly available to ALUs)
- **Key feature:** This is where computation actually happens. Instructions like FMA read their operands from registers and write results back to registers.
- **Tradeoff:** More registers per thread = fewer threads per SM (lower occupancy). The compiler manages register allocation, but CuTe DSL's `make_rmem_tensor` lets you explicitly allocate register-backed tensors.

## Data Movement Paths

Not all memory-to-memory paths are equal. The GPU hardware provides specialized instructions for certain paths:

```
  GMEM ──────────────────────────────────> RMEM     (LD.GLOBAL — load through L1/L2)
  GMEM ──────────────────> SMEM                     (CP.ASYNC — bypasses registers!)
                           SMEM ────────> RMEM      (LDS — load from shared memory)
                           SMEM ────────> RMEM      (LDMATRIX — warp-level structured load)
  GMEM <──────────────────────────────── RMEM       (ST.GLOBAL — store through L1/L2)
                           SMEM <──────── RMEM      (STS — store to shared memory)
```

The key insight is that **GMEM → SMEM can bypass registers entirely** using `cp.async` (Ampere+). This is important because registers are a scarce resource -- using them as a waypoint for data that's just passing through to SMEM is wasteful.

In CuTe DSL, each of these hardware paths is represented by a **CopyAtom** -- a type that encapsulates the instruction, its operand layout, and how threads cooperate to move data. Let's explore each path.

## Path 1: GMEM → RMEM → GMEM (Direct Load/Store)

The simplest data movement pattern: each thread loads data from global memory directly into its registers, computes on it, and stores results back. This is what happens when you index into a GMEM tensor inside a kernel.

Under the hood, the GPU issues `LD.GLOBAL` instructions that travel through L2 → L1 → registers. We don't need to explicitly manage SMEM at all.

Let's write a simple kernel that loads elements from GMEM into register-backed tensors using `make_rmem_tensor`, doubles them, and writes them back.

In [ ]:
ELEMS_PER_THREAD = 4

@cute.kernel
def gmem_to_rmem_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Each thread loads ELEMS_PER_THREAD elements from GMEM into registers,
    doubles them, and stores back to GMEM."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()

    global_tid = bidx * bdim + tidx

    # Allocate a register-backed tensor (RMEM) to hold this thread's data
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD, cutlass.Float32)

    # GMEM → RMEM: load elements into registers
    for i in range(ELEMS_PER_THREAD):
        rmem[i] = gIn[global_tid * ELEMS_PER_THREAD + i]

    # Compute in registers (doubling each element)
    for i in range(ELEMS_PER_THREAD):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM: store results back
    for i in range(ELEMS_PER_THREAD):
        gOut[global_tid * ELEMS_PER_THREAD + i] = rmem[i]


@cute.jit
def gmem_to_rmem(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD)
    gmem_to_rmem_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


# Test correctness
N = 1 << 20  # ~1M elements
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

gmem_to_rmem_fn = cute.compile(gmem_to_rmem, inp_, out_)
gmem_to_rmem_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

### What happened under the hood

1. **`cute.make_rmem_tensor(4, cutlass.Float32)`** allocated 4 FP32 values in registers -- this compiles down to 4 local variables in the PTX, each backed by a 32-bit register.

2. **`rmem[i] = gIn[...]`** generated `LD.GLOBAL` instructions. The data travels: GMEM → L2 → L1 → register file.

3. **`rmem[i] = rmem[i] * 2.0`** is a pure register operation (`FMUL`). Both operands and the result are in registers -- zero memory traffic.

4. **`gOut[...] = rmem[i]`** generated `ST.GLOBAL` instructions. Data travels: register file → L1 → L2 → GMEM.

This is the simplest pattern but also the least efficient for kernels where multiple threads need overlapping data -- each thread independently fetches from GMEM, wasting bandwidth on duplicate loads.

## Path 2: GMEM → SMEM → RMEM (The Two-Stage Pattern)

When threads within a block need overlapping data, we can save GMEM bandwidth by:
1. **Loading data from GMEM into SMEM** once (cooperatively across all threads in the block)
2. **Having each thread read from SMEM** into its registers

This is the bread-and-butter pattern for GEMM, convolution, reduction, and stencil kernels.

In the kernel below, we demonstrate this explicitly:
- Each thread cooperatively loads part of a block-sized chunk from GMEM into SMEM
- After a `__syncthreads()`, each thread reads from SMEM into registers
- The thread computes on registers and writes results back to GMEM

In [ ]:
BLOCK_SIZE = 256

@cute.kernel
def gmem_smem_rmem_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Demonstrates the GMEM → SMEM → RMEM → GMEM data movement pattern.
    Each thread block cooperatively loads a chunk into SMEM, then each thread
    reads its element from SMEM into a register, computes, and writes back."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Step 1: Allocate shared memory for the block (SMEM)
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, BLOCK_SIZE, alignment=16)
    smem = cute.make_tensor(smem_ptr, BLOCK_SIZE)

    # Step 2: GMEM → SMEM -- each thread loads one element cooperatively
    global_idx = bidx * BLOCK_SIZE + tidx
    smem[tidx] = gIn[global_idx]

    # Step 3: Synchronize -- ensure all threads have finished writing to SMEM
    cute.arch.sync_threads()

    # Step 4: SMEM → RMEM -- each thread reads its element into a register
    rmem = cute.make_rmem_tensor(1, cutlass.Float32)
    rmem[0] = smem[tidx]

    # Step 5: Compute in registers
    rmem[0] = rmem[0] * 2.0

    # Step 6: RMEM → GMEM -- write results back
    gOut[global_idx] = rmem[0]


@cute.jit
def gmem_smem_rmem(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    gmem_smem_rmem_kernel(mIn, mOut).launch(
        grid=(N // BLOCK_SIZE, 1, 1),
        block=(BLOCK_SIZE, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

gmem_smem_rmem_fn = cute.compile(gmem_smem_rmem, inp_, out_)
gmem_smem_rmem_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → SMEM → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

### Breaking down the SMEM pattern

The key CuTe DSL primitives for shared memory:

| Step | CuTe DSL Call | What it does |
|------|--------------|--------------|
| Allocate SMEM | `cute.arch.alloc_smem(dtype, num_elems, alignment)` | Statically allocates a block of shared memory, returns a pointer |
| Create SMEM tensor | `cute.make_tensor(smem_ptr, shape)` | Wraps the SMEM pointer with a layout to get an indexable tensor |
| GMEM → SMEM | `smem[tidx] = gIn[global_idx]` | Each thread loads one element (travels GMEM → L2 → L1 → RMEM → SMEM) |
| Synchronize | `cute.arch.sync_threads()` | Barrier -- ensures all threads have finished their SMEM writes |
| SMEM → RMEM | `rmem[0] = smem[tidx]` | Thread reads from SMEM into a register |

Note that the naive `smem[tidx] = gIn[idx]` path actually goes **GMEM → registers → SMEM** (two hops). The data briefly passes through registers because the basic load/store instructions require register operands. On Ampere+, `cp.async` can bypass this register waypoint -- we'll see that next.

## Path 3: GMEM → SMEM via `cp.async` (Register-Free Transfer)

Starting with Ampere (SM80), NVIDIA introduced `cp.async` -- an instruction that copies data directly from global memory to shared memory **without using registers as intermediaries**. This has two benefits:

1. **Saves registers:** The data never touches the register file, leaving more registers for computation.
2. **Asynchronous:** The copy is initiated and the thread can continue executing other instructions. The thread only waits when it actually needs the data in SMEM.

In CuTe DSL, this path uses `CopyG2SOp` (Copy Global-to-Shared Operation):

```python
op = cute.nvgpu.cpasync.CopyG2SOp()
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)
```

The `num_bits_per_copy` parameter controls the transaction size -- 128 bits means 4 FP32 elements per copy operation per thread. After issuing the copy, we call `cute.arch.cp_async_commit_group()` to commit the pending async copies and `cute.arch.cp_async_wait_group(0)` to wait for all pending groups to complete.

In [ ]:
BLOCK_SIZE_ASYNC = 256
ELEMS_PER_THREAD_ASYNC = 4
SMEM_SIZE = BLOCK_SIZE_ASYNC * ELEMS_PER_THREAD_ASYNC  # 1024 elements per block

@cute.kernel
def cp_async_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Demonstrates GMEM → SMEM via cp.async, then SMEM → RMEM → GMEM."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Allocate SMEM
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, SMEM_SIZE, alignment=16)
    smem = cute.make_tensor(smem_ptr, SMEM_SIZE)

    # Build the cp.async copy atom (32-bit per operation)
    # Note: 128-bit cp.async requires statically provable 16-byte alignment,
    # which needs proper tiling infrastructure (covered in a future notebook).
    # Here we use 32-bit granularity to demonstrate the async mechanism.
    cp_async_op = cute.nvgpu.cpasync.CopyG2SOp()
    cp_async_atom = cute.make_copy_atom(cp_async_op, cutlass.Float32, num_bits_per_copy=32)

    # Each thread is responsible for ELEMS_PER_THREAD_ASYNC contiguous elements
    base_idx = bidx * SMEM_SIZE + tidx * ELEMS_PER_THREAD_ASYNC
    smem_offset = tidx * ELEMS_PER_THREAD_ASYNC

    # GMEM → SMEM via cp.async (bypasses registers!)
    # Issue one 32-bit async copy per element
    for i in range(ELEMS_PER_THREAD_ASYNC):
        gmem_elem = cute.make_tensor(
            gIn.iterator + base_idx + i,
            cute.make_layout((1, 1), stride=(0, 1))
        )
        smem_elem = cute.make_tensor(
            smem_ptr + smem_offset + i,
            cute.make_layout((1, 1), stride=(0, 1))
        )
        cute.copy(cp_async_atom, gmem_elem, smem_elem)

    # Commit and wait for async copy to complete
    cute.arch.cp_async_commit_group()
    cute.arch.cp_async_wait_group(0)
    cute.arch.sync_threads()

    # SMEM → RMEM: load into registers for computation
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD_ASYNC, cutlass.Float32)
    for i in range(ELEMS_PER_THREAD_ASYNC):
        rmem[i] = smem[smem_offset + i]

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_ASYNC):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM
    for i in range(ELEMS_PER_THREAD_ASYNC):
        gOut[base_idx + i] = rmem[i]


@cute.jit
def cp_async_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    cp_async_kernel(mIn, mOut).launch(
        grid=(N // SMEM_SIZE, 1, 1),
        block=(BLOCK_SIZE_ASYNC, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

cp_async_fn = cute.compile(cp_async_demo, inp_, out_)
cp_async_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → SMEM (cp.async) → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

### cp.async vs naive GMEM → SMEM

The difference between the naive path and `cp.async`:

```
Naive:     GMEM  →  Registers  →  SMEM     (2 hops, uses registers as temporary)
cp.async:  GMEM  →  SMEM                   (1 hop, direct DMA by the memory subsystem)
```

**`cp.async` advantages:**
- **Frees up registers** -- data doesn't occupy register slots during transit
- **Asynchronous** -- the thread issues the copy and continues doing other work
- **Higher throughput** -- the memory controller can pipeline multiple 128-bit async copies

**The async handshake:**
1. `cute.copy_atom_call(cp_async_atom, src, dst)` -- initiates the async copy
2. `cute.arch.cp_async_commit_group()` -- commits all pending async copies into a "group"
3. `cute.arch.cp_async_wait_group(N)` -- waits until at most N groups are still in-flight (0 = wait for all)
4. `cute.arch.sync_threads()` -- ensures all threads have completed the wait before reading SMEM

## Using `autovec_copy` for Automatic Vectorization

CuTe DSL provides `autovec_copy` -- a copy function that analyzes the pointer alignment and layout of source and destination tensors to automatically select the widest safe vector width. Under the hood, it creates a `CopyUniversalOp` atom with the optimal `num_bits_per_copy`.

This is the simplest way to get vectorized copies between any memory spaces:

```python
cute.autovec_copy(src_tensor, dst_tensor)
```

`autovec_copy` examines:
1. The **layout alignment** (stride patterns) of both tensors
2. The **pointer alignment** (byte alignment of the base address)
3. Caps at 256 bits maximum

Let's use it to copy between GMEM and RMEM with automatic vectorization.

In [ ]:
ELEMS_PER_THREAD_VEC = 4

@cute.kernel
def autovec_kernel(gIn: cute.Tensor, gOut: cute.Tensor):
    """Uses autovec_copy for automatic vectorization of GMEM↔RMEM copies."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    global_tid = bidx * bdim + tidx
    base_idx = global_tid * ELEMS_PER_THREAD_VEC

    # Create a sub-tensor for this thread's GMEM slice
    # Using a 2D layout so autovec_copy can reason about vector width
    vec_layout = cute.make_layout((ELEMS_PER_THREAD_VEC, 1), stride=(1, 0))
    src = cute.make_tensor(gIn.iterator + base_idx, vec_layout)

    # Register-backed tensor with matching shape
    rmem = cute.make_rmem_tensor((ELEMS_PER_THREAD_VEC, 1), cutlass.Float32)

    # GMEM → RMEM: autovec_copy picks the widest safe vector width
    cute.autovec_copy(src, rmem)

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_VEC):
        rmem[i, 0] = rmem[i, 0] * 2.0

    # RMEM → GMEM
    dst = cute.make_tensor(gOut.iterator + base_idx, vec_layout)
    cute.autovec_copy(rmem, dst)


@cute.jit
def autovec_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD_VEC)
    autovec_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

autovec_fn = cute.compile(autovec_demo, inp_, out_)
autovec_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"autovec_copy GMEM → RMEM → GMEM: PASSED (N={N:,})")

### Why vectorization matters

Under the hood, `autovec_copy` selects the widest load/store instruction that the pointer alignment allows. Wider vector loads issue fewer instructions for the same data:

| Vector Width | PTX Instruction | Bytes/instruction | FP32 Elements |
|-------------|----------------|-------------------|---------------|
| 32-bit | `LD.GLOBAL.B32` | 4 | 1 |
| 64-bit | `LD.GLOBAL.B64` | 8 | 2 |
| 128-bit | `LD.GLOBAL.B128` | 16 | 4 |

Fewer instructions means less scheduling overhead and better memory bus utilization. The `autovec_copy` function handles this automatically based on the tensor's alignment properties.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Benchmark: autovec_copy (vectorized) vs scalar element-by-element
N = 1 << 22  # 4M elements
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)
inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

# Scalar kernel (1 element at a time, from Path 1)
scalar_fn = cute.compile(gmem_to_rmem, inp_, out_)
out.zero_()
scalar_fn(inp_, out_)
torch.testing.assert_close(out, inp * 2.0)

# Vectorized kernel (autovec_copy)
vec_fn = cute.compile(autovec_demo, inp_, out_)
out.zero_()
vec_fn(inp_, out_)
torch.testing.assert_close(out, inp * 2.0)

ms_scalar = triton.testing.do_bench(lambda: scalar_fn(inp_, out_), warmup=25, rep=100)
ms_vec = triton.testing.do_bench(lambda: vec_fn(inp_, out_), warmup=25, rep=100)

bytes_moved = 2 * N * 4
bw_scalar = bytes_moved / (ms_scalar * 1e-3) / 1e9
bw_vec = bytes_moved / (ms_vec * 1e-3) / 1e9

print(f"{'Method':>25} | {'Time (ms)':>10} | {'Bandwidth (GB/s)':>16}")
print("-" * 58)
print(f"{'Scalar (element-wise)':>25} | {ms_scalar:>10.4f} | {bw_scalar:>16.1f}")
print(f"{'autovec_copy (vectorized)':>25} | {ms_vec:>10.4f} | {bw_vec:>16.1f}")
print(f"{'Speedup':>25} | {ms_scalar / ms_vec:>10.2f}x |")

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(["Scalar\n(element-wise)", "autovec_copy\n(vectorized)"],
              [bw_scalar, bw_vec],
              color=["#4c72b0", "#c44e52"], edgecolor="black")
ax.set_ylabel("Effective Bandwidth (GB/s)")
ax.set_title(f"Scalar vs Vectorized Copy (N={N:,} FP32 elements)")
for bar, bw in zip(bars, [bw_scalar, bw_vec]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{bw:.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## Putting It All Together: Comparing All Paths

Let's benchmark all four data movement patterns side by side:

1. **Direct scalar (GMEM → RMEM → GMEM)** -- element-by-element loads/stores
2. **autovec_copy (GMEM → RMEM → GMEM)** -- automatically vectorized loads/stores
3. **SMEM staging naive (GMEM → SMEM → RMEM → GMEM)** -- cooperative load into SMEM via registers
4. **SMEM staging cp.async (GMEM → SMEM → RMEM → GMEM)** -- register-free GMEM→SMEM via async DMA

For this simple element-wise doubling, SMEM staging adds overhead without benefit (no data reuse). The real payoff comes in kernels like GEMM where multiple threads read overlapping data -- we include it here to compare the raw copy mechanics.

In [ ]:
sizes = [2**i for i in range(18, 25)]  # 256K to 16M elements

# Pre-compile all kernels at the largest size so compilation cost isn't in the benchmark
N_max = max(sizes)
inp_max = torch.randn(N_max, device="cuda", dtype=torch.float32)
out_max = torch.zeros(N_max, device="cuda", dtype=torch.float32)
inp_max_ = from_dlpack(inp_max, assumed_align=16)
out_max_ = from_dlpack(out_max, assumed_align=16)

direct_fn = cute.compile(gmem_to_rmem, inp_max_, out_max_)
smem_fn = cute.compile(gmem_smem_rmem, inp_max_, out_max_)
cpasync_fn = cute.compile(cp_async_demo, inp_max_, out_max_)
autovec_fn_ = cute.compile(autovec_demo, inp_max_, out_max_)

results = []
print(f"{'N':>12} | {'Direct (ms)':>12} | {'SMEM (ms)':>12} | {'cp.async (ms)':>14} | {'autovec (ms)':>13}")
print("-" * 72)

for N in sizes:
    inp = torch.randn(N, device="cuda", dtype=torch.float32)
    out = torch.zeros(N, device="cuda", dtype=torch.float32)
    inp_ = from_dlpack(inp, assumed_align=16)
    out_ = from_dlpack(out, assumed_align=16)

    ms_direct = triton.testing.do_bench(lambda: direct_fn(inp_, out_), warmup=25, rep=100)
    ms_smem = triton.testing.do_bench(lambda: smem_fn(inp_, out_), warmup=25, rep=100)
    ms_cpasync = triton.testing.do_bench(lambda: cpasync_fn(inp_, out_), warmup=25, rep=100)
    ms_autovec = triton.testing.do_bench(lambda: autovec_fn_(inp_, out_), warmup=25, rep=100)

    bytes_moved = 2 * N * 4  # read + write
    bw_direct = bytes_moved / (ms_direct * 1e-3) / 1e9
    bw_smem = bytes_moved / (ms_smem * 1e-3) / 1e9
    bw_cpasync = bytes_moved / (ms_cpasync * 1e-3) / 1e9
    bw_autovec = bytes_moved / (ms_autovec * 1e-3) / 1e9

    results.append((N, ms_direct, ms_smem, ms_cpasync, ms_autovec, bw_direct, bw_smem, bw_cpasync, bw_autovec))
    print(f"{N:>12,} | {ms_direct:>12.4f} | {ms_smem:>12.4f} | {ms_cpasync:>14.4f} | {ms_autovec:>13.4f}")

In [ ]:
labels = [f"2^{i}" for i in range(18, 25)]
bw_direct = [r[5] for r in results]
bw_smem = [r[6] for r in results]
bw_cpasync = [r[7] for r in results]
bw_autovec = [r[8] for r in results]

x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - 1.5*width, bw_direct, width, label="Direct (scalar)", color="steelblue", edgecolor="black")
ax.bar(x - 0.5*width, bw_autovec, width, label="autovec_copy (vectorized)", color="seagreen", edgecolor="black")
ax.bar(x + 0.5*width, bw_smem, width, label="SMEM staging (naive)", color="indianred", edgecolor="black")
ax.bar(x + 1.5*width, bw_cpasync, width, label="SMEM staging (cp.async)", color="goldenrod", edgecolor="black")

ax.set_xlabel("Input Size (N elements)")
ax.set_ylabel("Effective Bandwidth (GB/s)")
ax.set_title("Data Movement Path Comparison (FP32 element-wise x2)")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

### Analysis

For this simple **element-wise** kernel with no data reuse:
- **Direct GMEM → RMEM** should perform best -- it's the shortest path with vectorized 128-bit loads
- **SMEM staging (naive)** adds overhead: extra SMEM write, sync barrier, SMEM read -- all for no benefit since there's no data sharing between threads
- **cp.async staging** avoids the register waypoint but still has the SMEM detour

The SMEM-staging approaches shine in kernels with **data reuse** (like GEMM), where the cost of the SMEM round-trip is amortized over many register reads from the same shared data. In a future tiling notebook, we'll see how SMEM + cp.async enables software-pipelined GEMM kernels that approach peak hardware throughput.

## CuTe DSL Copy Atom Reference

CuTe DSL provides a rich set of **CopyAtom** types, each mapping to a specific hardware copy instruction. Below is a comprehensive catalog of every copy operation available in the `cutlass.cute` package.

### How Copy Atoms Work

A CopyAtom wraps a hardware instruction and defines:
- **The source and destination memory spaces** (GMEM, SMEM, RMEM, TMEM)
- **The data layout** each thread expects (how many elements, what pattern)
- **The vector width** (how many bits per copy instruction)

Usage pattern:
```python
# 1. Create a CopyOp (describes the hardware instruction)
op = cute.nvgpu.CopyUniversalOp()

# 2. Create a CopyAtom from the op (adds type and width info)
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)

# 3. Execute the copy on tensors with matching layout profile (V,)
cute.copy_atom_call(atom, src_tensor, dst_tensor)
```

### Copy Functions

| Function | Description |
|----------|-------------|
| `cute.copy(atom, src, dst)` | Main copy algorithm for tensors with layout profile `(V, Rest...)`. Handles recursive decomposition of multi-mode tensors. Supports predication via `pred=` kwarg. |
| `cute.copy_atom_call(atom, src, dst)` | Low-level single-atom copy for tensors with layout profile `(V,)`. No recursion -- directly issues one copy instruction. |
| `cute.basic_copy(src, dst)` | Simple element-wise copy. No atom needed -- uses SIMT sync copy internally. |
| `cute.basic_copy_if(pred, src, dst)` | Predicated element-wise copy. Copies `src[i]` to `dst[i]` only where `pred[i]` is true. |
| `cute.autovec_copy(src, dst)` | Auto-vectorizing copy. Analyzes layout alignment to pick the widest safe vector width automatically. |
| `cute.prefetch(atom, src)` | Prefetches data from GMEM into L2 cache. Currently only supports TMA prefetch atoms. |

### 1. Universal Copy — `cute.nvgpu.CopyUniversalOp`

**Path:** Any → Any (GMEM↔RMEM, SMEM↔RMEM, GMEM↔SMEM via registers)
**Architecture:** All (SM50+)
**PTX:** `LD.GLOBAL`, `LD.SHARED`, `ST.GLOBAL`, `ST.SHARED`, etc.

The general-purpose copy. Maps to standard load/store instructions. Supports vectorization up to 256 bits.

```python
op = cute.nvgpu.CopyUniversalOp()
atom = cute.make_copy_atom(
    op,
    cutlass.Float32,             # element type (determines layout)
    num_bits_per_copy=128,       # vectorization: 32, 64, 128, or 256 bits (0 = auto)
    l1c_evict_priority=cute.nvgpu.CacheEvictionPriority.EVICT_NORMAL,  # L1 cache hint
    memory_order=cute.nvgpu.MemoryOrder.WEAK,          # memory ordering
    memory_scope=cute.nvgpu.MemoryScope.CTA,           # visibility scope
    invariant=False,             # True = use read-only cache (LDG)
)
```

**Parameters:**
| Parameter | Options | Description |
|-----------|---------|-------------|
| `num_bits_per_copy` | 0, 32, 64, 128, 256 | Bits per copy instruction. 0 = auto-vectorize |
| `l1c_evict_priority` | `EVICT_NORMAL`, `EVICT_FIRST`, `EVICT_LAST`, `EVICT_UNCHANGED`, `NO_ALLOCATE` | L1 cache eviction hint |
| `memory_order` | `WEAK`, `RELAXED`, `ACQUIRE`, `RELEASE`, `ACQ_REL`, `SC`, `MMIO`, `CONSTANT`, `VOLATILE` | Memory ordering semantics |
| `memory_scope` | `CTA`, `CLUSTER`, `GPU`, `SYS` | Visibility scope for ordering |
| `invariant` | `True`/`False` | Read-only optimization (texture cache path) |

### 2. Asynchronous GMEM → SMEM — `cute.nvgpu.cpasync.CopyG2SOp`

**Path:** GMEM → SMEM (bypasses registers)
**Architecture:** SM80+ (Ampere)
**PTX:** `cp.async`

Initiates an asynchronous copy from global memory directly to shared memory. The thread does not wait for completion -- you must explicitly commit and wait.

```python
op = cute.nvgpu.cpasync.CopyG2SOp(
    cache_mode=cute.nvgpu.cpasync.LoadCacheMode.ALWAYS  # L1 caching policy
)
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)
```

**Cache modes:**
| Mode | Description |
|------|-------------|
| `ALWAYS` | Cache in L1 (default) |
| `GLOBAL` | Cache in L2 only, bypass L1 |
| `STREAMING` | Streaming access, evict first |
| `LAST_USE` | Hint that this is the last access |
| `NONE` | No caching |

**Synchronization protocol:**
```python
cute.copy_atom_call(cp_async_atom, gmem_src, smem_dst)  # initiate
cute.arch.cp_async_commit_group()                         # commit pending copies
cute.arch.cp_async_wait_group(0)                          # wait for all groups
cute.arch.sync_threads()                                  # block-wide barrier
```

### 3. TMA Bulk Tensor Copy — `cute.nvgpu.cpasync.CopyBulkTensorTile*`

**Architecture:** SM90+ (Hopper)
**PTX:** `cp.async.bulk.tensor`

The **Tensor Memory Accelerator (TMA)** is a dedicated hardware unit on Hopper+ GPUs that can copy entire multi-dimensional tiles between GMEM and SMEM. Unlike `cp.async` (which is thread-initiated), TMA operations are issued by a single thread and the hardware handles the entire tile transfer.

| CopyOp | Path | Description |
|--------|------|-------------|
| `CopyBulkTensorTileG2SOp` | GMEM → SMEM | Bulk tensor tile load |
| `CopyBulkTensorTileG2SMulticastOp` | GMEM → SMEM (multicast) | Load + broadcast to multiple CTAs in a cluster |
| `CopyBulkTensorTileS2GOp` | SMEM → GMEM | Bulk tensor tile store |
| `CopyReduceBulkTensorTileS2GOp` | SMEM → GMEM (reduce) | Store with atomic reduction (ADD, MIN, MAX, etc.) |

```python
# GMEM → SMEM via TMA
op = cute.nvgpu.cpasync.CopyBulkTensorTileG2SOp(
    cta_group=cute.nvgpu.tcgen05.CtaGroup.ONE  # ONE or TWO (2-CTA cooperative)
)

# TMA requires a tensor map descriptor — built via make_tiled_tma_atom
tma_atom, tma_tensor = cute.nvgpu.cpasync.make_tiled_tma_atom(
    op, gmem_tensor, smem_layout, cta_tiler
)

# Execute with mbarrier synchronization
cute.copy(tma_atom, src, dst, tma_bar_ptr=mbar_ptr, mcast_mask=mask)
```

**TMA key features:**
- Single-thread issue: only one thread needs to initiate the copy
- Hardware handles address computation for multi-dimensional tiles
- Supports multicast to multiple CTAs in a cluster (SM90+)
- Uses mbarrier for synchronization instead of cp.async groups

### 4. Bulk Copy (Non-Tensor) — `CopyBulk*Op`

**Architecture:** SM90+ (Hopper)
**PTX:** `cp.async.bulk`

Lower-level bulk copy operations that work with raw addresses (not tensor descriptors). These are used internally by CUTLASS but are not part of the public API (`__all__`).

| CopyOp | Path | Description |
|--------|------|-------------|
| `CopyBulkG2SOp` | GMEM → SMEM | Bulk raw copy |
| `CopyBulkG2SMulticastOp` | GMEM → SMEM | Bulk raw copy with multicast |
| `CopyBulkS2GOp` | SMEM → GMEM | Bulk raw store |
| `CopyBulkS2GByteMaskOp` | SMEM → GMEM | Bulk store with per-byte mask (SM100+) |
| `CopyBulkS2SOp` | SMEM → SMEM (cross-CTA) | Copy between CTAs in a cluster |
| `CopyDsmemStoreOp` | RMEM → DSMEM | Async store to distributed shared memory |

### 5. Warp-Level Matrix Load/Store — `cute.nvgpu.warp.LdMatrix*` / `StMatrix*`

**Path:** SMEM ↔ RMEM (structured for Tensor Core consumption)
**Architecture:** SM75+ (Turing)
**PTX:** `ldmatrix`, `stmatrix`

These are **warp-level** instructions: all 32 threads in a warp cooperate to load/store a matrix tile from SMEM into registers in the exact layout that Tensor Cores expect. This avoids expensive register shuffles before MMA instructions.

#### Load Matrix (SMEM → RMEM)

| CopyOp | Matrix Shape | Element Size | Notes |
|--------|-------------|-------------|-------|
| `LdMatrix8x8x16bOp` | 8x8 | 16-bit | Basic matrix load, `.m8n8` qualifier |
| `LdMatrix8x16x8bOp` | 8x16 | 8-bit | Supports 4-bit and 6-bit unpacking |
| `LdMatrix16x8x8bOp` | 16x8 | 8-bit | Transpose required, lowers to `.m16n16` + permutation |
| `LdMatrix16x16x8bOp` | 16x16 | 8-bit | Transpose + optional unpacking (4b, 6b) |

```python
op = cute.nvgpu.warp.LdMatrix8x8x16bOp(
    transpose=False,     # whether to transpose the loaded matrix
    num_matrices=4,      # how many matrices to load (1, 2, or 4)
)
atom = cute.make_copy_atom(op, cutlass.Float16)
```

#### Store Matrix (RMEM → SMEM)

| CopyOp | Matrix Shape | Element Size | Notes |
|--------|-------------|-------------|-------|
| `StMatrix8x8x16bOp` | 8x8 | 16-bit | Basic matrix store |
| `StMatrix16x8x8bOp` | 16x8 | 8-bit | Transpose store |

```python
op = cute.nvgpu.warp.StMatrix8x8x16bOp(
    transpose=False,
    num_matrices=4,
)
atom = cute.make_copy_atom(op, cutlass.Float16)
```

**When to use:** Before/after MMA (matrix multiply-accumulate) operations. The `ldmatrix`/`stmatrix` instructions are designed to load data from SMEM into registers in exactly the layout that `mma.sync` instructions expect, avoiding costly register shuffles.

### 6. TCGen05 — Tensor Core Gen 0.5 Operations (SM100+/Blackwell)

**Path:** TMEM ↔ RMEM, SMEM → TMEM
**Architecture:** SM100+ (Blackwell)
**PTX:** `tcgen05.ld`, `tcgen05.st`, `tcgen05.cp`

Blackwell introduces **Tensor Memory (TMEM)** -- a new memory space dedicated to Tensor Core operands. TCGen05 operations move data between TMEM, registers, and shared memory.

#### TMEM Load (TMEM → RMEM)

| CopyOp | Shape | Notes |
|--------|-------|-------|
| `Ld16x64bOp` | 16x64-bit | |
| `Ld16x128bOp` | 16x128-bit | |
| `Ld16x256bOp` | 16x256-bit | |
| `Ld16x32bx2Op` | 16x32-bit x2 | |
| `Ld32x32bOp` | 32x32-bit | |

Parameters: `repeat` (`Repetition.x1`, `x2`, `x4`, etc.), `pack` (`Pack.NONE`, etc.)

#### TMEM Load with Reduction

| CopyOp | Shape | Notes |
|--------|-------|-------|
| `LdRed16x32bx2Op` | 16x32-bit x2 | Built-in reduction (MAX, MIN, ADD, etc.) |
| `LdRed32x32bOp` | 32x32-bit | Built-in reduction |

#### TMEM Store (RMEM → TMEM)

| CopyOp | Shape | Notes |
|--------|-------|-------|
| `St16x64bOp` | 16x64-bit | |
| `St16x128bOp` | 16x128-bit | |
| `St16x256bOp` | 16x256-bit | |
| `St16x32bx2Op` | 16x32-bit x2 | |
| `St32x32bOp` | 32x32-bit | |

Parameters: `repeat` (required), `unpack` (`Unpack.NONE`, etc.)

#### SMEM → TMEM Copy

| CopyOp | Shape | Notes |
|--------|-------|-------|
| `Cp128x256bOp` | 128x256-bit | SM100 only |
| `Cp128x128bOp` | 128x128-bit | SM100 only |
| `Cp4x256bOp` | 4x256-bit | SM100 only |
| `Cp4x32x128bOp` | 32x128-bit | With warpx4 broadcast |
| `Cp2x64x128b0213Op` | 64x128-bit | With warpx2::02_13 broadcast |
| `Cp2x64x128b0123Op` | 64x128-bit | With warpx2::01_23 broadcast |

### Summary: Which CopyAtom for Which Path?

| Path | Best CopyAtom | Architecture | Use Case |
|------|--------------|-------------|----------|
| **GMEM → RMEM** | `CopyUniversalOp` | All | Element-wise ops, simple loads |
| **RMEM → GMEM** | `CopyUniversalOp` | All | Writing results back |
| **GMEM → SMEM** | `CopyG2SOp` (cp.async) | SM80+ | Staging data for block-wide reuse |
| **GMEM → SMEM** | `CopyBulkTensorTileG2SOp` (TMA) | SM90+ | Large tile loads, GEMM |
| **SMEM → GMEM** | `CopyBulkTensorTileS2GOp` (TMA) | SM90+ | Tile stores |
| **SMEM → RMEM** | `CopyUniversalOp` | All | General SMEM reads |
| **SMEM → RMEM** | `LdMatrix*Op` (ldmatrix) | SM75+ | Loading MMA operands |
| **RMEM → SMEM** | `CopyUniversalOp` | All | General SMEM writes |
| **RMEM → SMEM** | `StMatrix*Op` (stmatrix) | SM75+ | Storing MMA results |
| **SMEM → TMEM** | `Cp*Op` (tcgen05) | SM100+ | Blackwell Tensor Core operands |
| **TMEM ↔ RMEM** | `Ld*/St*Op` (tcgen05) | SM100+ | Blackwell Tensor Core results |

The evolution across GPU generations is clear:
- **Turing (SM75):** Introduced `ldmatrix`/`stmatrix` for structured SMEM↔RMEM transfers
- **Ampere (SM80):** Added `cp.async` for register-free GMEM→SMEM
- **Hopper (SM90):** Added TMA for hardware-managed multi-dimensional tile transfers
- **Blackwell (SM100):** Added TMEM as a new memory tier dedicated to Tensor Cores